# 08 - SHAP Explainability & Irrigation Recommendation

Computes TreeSHAP feature contributions and generates transparent decision-support irrigation recommendations.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.data_loader import load_all_raw_data
from src.preprocessing import combine_and_clean_datasets
from src.hampel_filter import run_hampel_pipeline
from src.ceemdan import run_ceemdan_pipeline
from src.reconstruction import run_reconstruction_pipeline
from src.feature_engineering import generate_features
from src.model import chronological_train_val_test_split, train_lightgbm_model
from src.explainability import run_shap_analysis
from src.recommendation import generate_irrigation_advisories

_, df_clean = combine_and_clean_datasets(*load_all_raw_data())
df_hampel = run_hampel_pipeline(df_clean)
df_imfs, imfs = run_ceemdan_pipeline(df_hampel)
df_recon, _, _ = run_reconstruction_pipeline(df_imfs, imfs)
df_ready, feature_cols, target = generate_features(df_recon)

splits = chronological_train_val_test_split(df_ready, feature_cols, target)
model = train_lightgbm_model(splits['train'][0], splits['train'][1], splits['val'][0], splits['val'][1])

# SHAP Analysis
run_shap_analysis(model, splits['test'][0])

# Irrigation Recommendation
test_preds = model.predict(splits['test'][0])
advisories = generate_irrigation_advisories(splits['test'][2], test_preds)


In [ ]:
advisories.head(10)
